In [12]:
def get_ndvi_area(lat, lon, start_date, end_date):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    
    # 1. Cargar colección
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(region) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    
    # 2. Verificar si hay imágenes
    count = s2.size().getInfo()
    if count == 0:
        return "No hay imágenes disponibles para este periodo y filtro de nubes"
    
    # 3. Calcular NDVI
    def add_ndvi(image):
        return image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    # 4. Procesar y reducir
    ndvi_col = s2.map(add_ndvi)
    mean_ndvi = ndvi_col.mean()
    
    stats = mean_ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=10,
        bestEffort=True
    )
    
    # 5. Extracción segura
    # Convertimos a diccionario y verificamos si la llave existe
    result = stats.getInfo()
    return result.get('NDVI', 'No se pudo calcular el promedio de NDVI')

In [13]:
start_date = '2025-01-01'
end_date = '2026-06-27'
ndvi_area = get_ndvi_area(7.3297, -73.1867, start, end)

print(ndvi_area)

No hay imágenes disponibles para este periodo y filtro de nubes


In [14]:
def check_image_inventory(lat, lon, start_date, end_date):
    point = ee.Geometry.Point([lon, lat])
    # Colección completa sin filtros restrictivos
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(point) \
        .filterDate(start_date, end_date)
    
    # Obtener lista de IDs y fechas
    count = s2.size().getInfo()
    if count > 0:
        # Obtenemos los IDs de las primeras 5 imágenes para inspeccionar
        sample = s2.limit(5).reduceColumns(ee.Reducer.toList(), ['system:time_start']).get('list').getInfo()
        return f"Imágenes encontradas: {count}. Fechas (timestamp): {sample}"
    else:
        return "No hay ninguna imagen de Sentinel-2 en esta ubicación para este rango de fechas."

# Ejecutar diagnóstico
print(check_image_inventory(7.3297, -73.1867, '2025-01-01', '2026-06-27'))

Imágenes encontradas: 297. Fechas (timestamp): [1735831857947, 1735831854548, 1736263858458, 1736263855070, 1736695855432]


In [20]:
def get_ndvi_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            # Definir meses del trimestre
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            # Filtrar imágenes Sentinel-2
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate(f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-28')
            
            # Función NDVI
            def add_ndvi(image):
                return image.normalizedDifference(['B8', 'B4']).rename('NDVI')
            
            # Calcular mediana trimestral
            ndvi_med = s2.map(add_ndvi).median()
            
            # Extraer valor promedio del área
            stats = ndvi_med.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=10,
                bestEffort=True
            ).get('NDVI')
            
            # Guardar resultado (usando .getInfo() con manejo de errores)
            try:
                val = stats.getInfo()
                report[str(year)][f"Q{q}"] = val if val is not None else 0
            except:
                report[str(year)][f"Q{q}"] = 0
                
    return report

# Ejecutar con el rango donde sabemos que hay datos
start = 2020
end = 2026
ndvi_val = get_ndvi_by_year_and_quarter(7.3297, -73.1867, start, end)

import json
print(json.dumps(ndvi_val, indent=4))

{
    "2020": {
        "Q1": 0.6806555782301085,
        "Q2": 0.2434579682190943,
        "Q3": 0.28592839296050193,
        "Q4": 0.7350524443844915
    },
    "2021": {
        "Q1": 0.7254982630691623,
        "Q2": 0.26414197976614406,
        "Q3": 0.3543036844509362,
        "Q4": 0.2447838506395563
    },
    "2022": {
        "Q1": 0.7604506734165835,
        "Q2": 0.6022565879057842,
        "Q3": 0.7042222677657237,
        "Q4": 0.3749842051241771
    },
    "2023": {
        "Q1": 0.7224360223200426,
        "Q2": 0.4734144660268068,
        "Q3": 0.42856136261556965,
        "Q4": 0.43149771115421337
    },
    "2024": {
        "Q1": 0.6664719453058198,
        "Q2": 0.14026554493961726,
        "Q3": 0.6848189316985139,
        "Q4": 0.538541972326514
    },
    "2025": {
        "Q1": 0.7871039634186136,
        "Q2": 0.33324353085536706,
        "Q3": 0.7020437941928293,
        "Q4": 0.7473026431385718
    },
    "2026": {
        "Q1": 0.2923392609015586,
        "

In [21]:
def get_vegetation_indices_by_year_and_quarter(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate(f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-28')
            
            def add_indices(image):
                ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
                ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
                ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
                # EVI requiere bandas B8(NIR), B4(Red), B2(Blue)
                evi = image.expression(
                    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
                        'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')
                    }).rename('EVI')
                # BSI requiere B4(Red), B11(SWIR1), B8(NIR), B2(Blue)
                bsi = image.expression(
                    '((RED + SWIR1) - (NIR + BLUE)) / ((RED + SWIR1) + (NIR + BLUE))', {
                        'RED': image.select('B4'), 'SWIR1': image.select('B11'),
                        'NIR': image.select('B8'), 'BLUE': image.select('B2')
                    }).rename('BSI')
                return image.addBands([ndvi, ndre, ndmi, evi, bsi])
            
            indices_med = s2.map(add_indices).median()
            
            stats = indices_med.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=10,
                bestEffort=True
            ).getInfo()
            
            report[str(year)][f"Q{q}"] = stats
                
    return report

In [22]:
import json

# Define los parámetros
lat = 7.3297
lon = -73.1867
start = 2025
end = 2026

# Llama a la función
vegetation_data = get_vegetation_indices_by_year_and_quarter(lat, lon, start, end)

# Imprime el resultado de forma legible
print(json.dumps(vegetation_data, indent=4))

{
    "2025": {
        "Q1": {
            "AOT": 184.8765705953526,
            "B1": 410.8206806688792,
            "B11": 1742.2970403003126,
            "B12": 847.8418080848816,
            "B2": 376.18054478329645,
            "B3": 564.7437563987216,
            "B4": 355.533661154717,
            "B5": 991.4508578165226,
            "B6": 2565.938168957278,
            "B7": 3089.5956783420697,
            "B8": 3108.489467316102,
            "B8A": 3393.4015139763574,
            "B9": 3247.875903577077,
            "BSI": -0.23438068488406752,
            "EVI": 2.3519157393981565,
            "MSK_CLASSI_CIRRUS": 0,
            "MSK_CLASSI_OPAQUE": 0,
            "MSK_CLASSI_SNOW_ICE": 0,
            "MSK_CLDPRB": 0,
            "MSK_SNWPRB": 0,
            "NDMI": 0.2706133596155528,
            "NDRE": 0.5059245569543677,
            "NDVI": 0.7871039634186136,
            "QA10": null,
            "QA20": null,
            "QA60": 0,
            "SCL": 4,
            "TC

In [23]:
def get_clean_vegetation_indices(lat, lon, start_year, end_year):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(56).bounds()
    report = {}
    
    # Lista de índices que queremos
    indices_to_keep = ['NDVI', 'EVI', 'NDMI', 'BSI', 'NDRE']

    for year in range(start_year, end_year + 1):
        report[str(year)] = {}
        for q in range(1, 5):
            start_month = (q - 1) * 3 + 1
            end_month = q * 3
            
            s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate(f'{year}-{start_month:02d}-01', f'{year}-{end_month:02d}-28')
            
            def add_indices(image):
                ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
                ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
                ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
                evi = image.expression(
                    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
                        'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')
                    }).rename('EVI')
                bsi = image.expression(
                    '((RED + SWIR1) - (NIR + BLUE)) / ((RED + SWIR1) + (NIR + BLUE))', {
                        'RED': image.select('B4'), 'SWIR1': image.select('B11'),
                        'NIR': image.select('B8'), 'BLUE': image.select('B2')
                    }).rename('BSI')
                
                # Retornamos solo los índices calculados
                return image.addBands([ndvi, ndre, ndmi, evi, bsi]).select(indices_to_keep)
            
            # Calculamos la mediana de la colección filtrada
            indices_med = s2.map(add_indices).median()
            
            # Reducción espacial
            stats = indices_med.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=10,
                bestEffort=True
            ).getInfo()
            
            # Asignar resultados (si stats está vacío, llenamos con 0)
            report[str(year)][f"Q{q}"] = {k: stats.get(k, 0) for k in indices_to_keep}
                
    return report

In [ ]:
import json

# Define los parámetros
lat = 7.3297
lon = -73.1867
start = 2020
end = 2026

# Llama a la función
vegetation_data = get_clean_vegetation_indices(lat, lon, start, end)

# Imprime el resultado de forma legible
print(json.dumps(vegetation_data, indent=4))

{
    "2020": {
        "Q1": {
            "NDVI": 0.6806555782301085,
            "EVI": 2.2475156913792005,
            "NDMI": 0.25384318953125085,
            "BSI": -0.2107604319829029,
            "NDRE": 0.47402390402372196
        },
        "Q2": {
            "NDVI": 0.2434579682190943,
            "EVI": -0.10251998745838169,
            "NDMI": 0.2706858740234612,
            "BSI": -0.184749234824521,
            "NDRE": 0.17766225262165755
        },
        "Q3": {
            "NDVI": 0.28592839296050193,
            "EVI": 0.3377652583949619,
            "NDMI": 0.27642764429351824,
            "BSI": -0.20033816116354994,
            "NDRE": 0.17847313344505414
        },
        "Q4": {
            "NDVI": 0.7350524443844915,
            "EVI": 2.288858350318132,
            "NDMI": 0.2636538799656235,
            "BSI": -0.20899167499232713,
            "NDRE": 0.4999164914582227
        }
    },
    "2021": {
        "Q1": {
            "NDVI": 0.7254982630691623,


In [7]:
import ee
from datetime import datetime, timedelta

def mask_s2_clouds(image):
    """
    Máscara de nubes y cirros para Sentinel-2 usando la banda QA60.
    """
    qa = image.select('QA60')

    # Bits 10 y 11 son nubes y cirros respectivamente
    cloud_bit_mask = 10
    cirrus_bit_mask = 11

    # Bits que deben ser 0 para que el píxel sea válido
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))

    # Aplicar la máscara y escalar los valores de reflectancia (de 0-10000 a 0-1)
    return image.updateMask(mask).divide(10000)

def get_sentinel2_ndvi_stats(lat, lon, buffer_meters=118):
    """
    Calcula el NDVI medio en un área de ~1 ha (radio ~118m) 
    utilizando Sentinel-2 para los últimos 6 meses.
    
    Args:
        lat (float): Latitud.
        lon (float): Longitud.
        buffer_meters (float): Radio del buffer (118m aprox 1 hectá$\\text{áre}$).
        
    Returns:
        float: Valor medio de NDVI, o None si no hay imágenes disponibles.
    """
    try:
        # 1. Definir punto y área de interés (ROI)
        point = ee.Geometry.Point([lon, lat])
        roi = point.buffer(buffer_meters)

        # 2. Definir rango de fechas (últimos 180 días)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=180)
        
        start_date_str = start_date.strftime('%Y-%m-%d')
        end_date_str = end_date.strftime('%Y-%m-%d')

        # 3. Cargar y filtrar colección Sentinel-2 L2A (Surface Reflectance)
        # Usamos la colección HARMONIZED para consistencia temporal
        s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(roi)
                  .filterDate(start_date_str, end_date_str)
                  # Filtro previo por metadatos: solo imágenes con < 20% de nubes
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                  .map(mask_s2_clouds))

        # Verificar si hay imágenes en la colección
        count = s2_col.size().getInfo()
        if count == 0:
            print(f"No se encontraron imágenes para el periodo {start_date_str} a {end_date_str}")
            return None

        # 4. Crear un compuesto de Mediana (Median Composite)
        # La mediana es robusta frente a nubes residuales y sombras
        composite = s2_col.median().clip(roi)

        # 5. Calcular NDVI: (B8 - B4) / (B8 + B4)
        # B8 = NIR (Near Infrared), B4 = Red
        ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

        # 6. Reducción espacial: Calcular la media de todos los píxeles en el ROI
        stats = ndvi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=10,  # Resolución nativa de Sentinel-2 (10m)
            maxPixels=1e9
        )

# ----

        try:
            # 2. ¡EL PASO CLAVE! Convertir el objeto del servidor a un diccionario de Python
            # .getInfo() descarga los datos y los convierte en un dict de Python real
            stats_dict = stats.getInfo() 
            
            # 3. Ahora stats_dict es un dict de Python, podemos usar .get() con seguridad
            # El nombre de la clave será el nombre de la banda (ej: 'ndvi')
            # Buscamos 'ndvi' o cualquier clave que contenga 'ndvi'
            value = None
            for key in stats_dict.keys():
                if 'ndvi' in key.lower():
                    value = stats_dict[key]
                    break
            
            return value

        except Exception as e:
            print(f"Error al extraer valor: {e}")
            return None

# ----
        # return stats.get('nd').getInfo() if 'nd' in stats else stats.get('ndvi', None)
        # Nota: El nombre de la banda puede variar según el proceso, 
        # pero tras el proceso de reduccion suele ser 'nd' o 'ndvi'
        
    except Exception as e:
        print(f"Error en el procesamiento: {e}")
        return None


In [8]:
# --- Ejemplo de Uso ---
if __name__ == "__main__":
    # Tu código aquí
    lat_test, lon_test = 7.3297, -71.1867 # ejemplo
    resultado = get_sentinel2_ndvi_stats(lat_test, lon_test)
    print(resultado)

0.7566806561540469
